# Titanic Survival Analysis – Data Analyst Project
**Tools:** Python, pandas

## 1. Business problem
The objective of this project is to clean the raw Titanic passenger dataset and find relevant information on **what factors influenced passenger survivability**.

**Questions we will answer**
1. How many male and female passengers survived, and what was the average age of survivors?
2. How did survival rates differ across embarkation ports (`S`, `C`, and `Q`)?
3. How did survivability compare across key demographic groups (children `< 15`, elderly `> 60`, middle-aged men `30–45`, and middle-aged women `30–45`)?
4. How did traveling solo versus traveling with family—and having a family size above or below `3`—affect survival?

> The analysis uses **pure pandas** for data inspection, cleaning, feature engineering, and exploratory data analysis.

## 2. Import libraries

In [ ]:
import pandas as pd

## 3. Load and inspect the data

In [ ]:
# 1. Load the CSV file
df = pd.read_csv("Titanic-Dataset.csv")

# 2. Initial Inspection
print("Before cleaning:", df.shape)
print("Columns:", df.columns.tolist())
print("Data types before:\n", df.dtypes)
print("\nMissing values:\n", df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
df.head()

**What we found:** missing values in `Age`, `Embarked`, and `Cabin`, potential duplicate rows or stray whitespace in text fields, and `Age` stored as float values. We clean these next.

## 4. Clean the data

In [ ]:
# 3. Clean column names and text spacing
df = df.drop(columns=["Unnamed: 0"], errors="ignore")
df.columns = df.columns.str.strip()

text_cols = df.select_dtypes(include="object").columns
for col in text_cols:
    df[col] = df[col].str.strip()

# 4. Remove duplicates (keeping first occurrence)
df = df.drop_duplicates(keep="first")

# 5. Fill missing values
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])
df["Cabin"] = df["Cabin"].fillna("Unknown")

# 6. Keep only valid values
df = df[df["Age"] > 0]
df = df[df["Fare"] >= 0]
df = df[df["Survived"].isin([0, 1])]
df = df[df["Pclass"].isin([1, 2, 3])]

# 7. Type Conversion
df["Pclass"] = df["Pclass"].astype("category")
df["Age"] = df["Age"].astype("int64")

print("After cleaning:", df.shape)
print("Missing values left:\n", df.isna().sum())

## 5. Create new columns (feature engineering)

In [ ]:
# 8. Feature Engineering: Create family_size column and drop SibSp & Parch
df["family_size"] = df["SibSp"] + df["Parch"] + 1
df.drop(columns=["SibSp", "Parch"], inplace=True)

# 9. Reset index and verify final dataset
df = df.reset_index(drop=True)

print("--- Final Dataset Summary ---")
print("Final columns:", df.columns.tolist())
print("Data types after:\n", df.dtypes)
df[["Name", "Age", "Sex", "family_size"]].head()

## 6. Exploratory data analysis (EDA)

### Q1. Survivors by gender and average age of survivors

In [ ]:
print("--- Survival Analysis ---")

# Filter only passengers who survived (Survived == 1)
survivors = df[df["Survived"] == 1]

# Count how many males and females survived
survival_by_sex = survivors["Sex"].value_counts()
print("Survivors by Gender:\n", survival_by_sex)

# Print them individually as clean numbers
male_survived = len(survivors[survivors["Sex"] == "male"])
female_survived = len(survivors[survivors["Sex"] == "female"])
print(f"\nMales survived: {male_survived}")
print(f"Females survived: {female_survived}")

# Average age of survivors (vs. non-survivors)
avg_survivor_age = survivors["Age"].mean()
print(f"Average age of survivors: {avg_survivor_age:.2f} years")

# Average age grouped by Survived (0 = Did not survive, 1 = Survived)
print("\nAverage age by survival status:\n", df.groupby("Survived")["Age"].mean())

### Q2. Survival rates by embarked port (`S`, `C`, `Q`)

In [ ]:
print("--- Survival by Embarked Port ---")

# Survival rate as a percentage (0% to 100%)
embarked_rate = df.groupby("Embarked")["Survived"].mean() * 100
print("Survival Rate (%) by Port:\n", embarked_rate.round(2))

# Detailed breakdown: Total passengers, Survivors, and Survival %
embarked_summary = df.groupby("Embarked")["Survived"].agg(
    Total_Passengers="count",
    Survived_Count="sum",
    Survival_Rate_Pct=lambda x: round(x.mean() * 100, 2)
)
print("\nDetailed Breakdown:\n", embarked_summary)

### Q3. Survival rate by age and gender groups

In [ ]:
print("--- Survival Rate by Age & Gender Groups ---")

# 1. Children (Age < 15)
children = df[df["Age"] < 15]

# 2. Elderly (Age > 60)
elderly = df[df["Age"] > 60]

# 3. Middle-aged men (Age 30 to 45, male)
mid_men = df[(df["Age"] >= 30) & (df["Age"] <= 45) & (df["Sex"] == "male")]

# 4. Middle-aged women (Age 30 to 45, female)
mid_women = df[(df["Age"] >= 30) & (df["Age"] <= 45) & (df["Sex"] == "female")]

# Print survival rates (%) and counts
print(f"Children (< 15)             : {children['Survived'].mean() * 100:.2f}% ({children['Survived'].sum()}/{len(children)})")
print(f"Elderly (> 60)              : {elderly['Survived'].mean() * 100:.2f}% ({elderly['Survived'].sum()}/{len(elderly)})")
print(f"Middle-aged Men (30-45)     : {mid_men['Survived'].mean() * 100:.2f}% ({mid_men['Survived'].sum()}/{len(mid_men)})")
print(f"Middle-aged Women (30-45)   : {mid_women['Survived'].mean() * 100:.2f}% ({mid_women['Survived'].sum()}/{len(mid_women)})")

### Q4. Solo travelers vs. family travelers and family size threshold (`3`)

In [ ]:
# Travelling with family vs solo
print("--- Solo Travelers vs. With Family ---")
solo = df[df["family_size"] == 1]
with_family = df[df["family_size"] > 1]

print(f"Solo travelers survival rate: {solo['Survived'].mean() * 100:.2f}%")
print(f"With family survival rate: {with_family['Survived'].mean() * 100:.2f}%")

# Survival Rate: Family Size Above 3 vs. Below 3
print("\n--- Survival Rate by Family Size (Threshold: 3) ---")
print(f"Average family size: {df['family_size'].mean():.2f}")

above_3 = df[df["family_size"] > 3]
below_3 = df[df["family_size"] < 3]
equal_3 = df[df["family_size"] == 3]

print(f"Family size > 3  : {above_3['Survived'].mean() * 100:.2f}% ({above_3['Survived'].sum()}/{len(above_3)})")
print(f"Family size < 3  : {below_3['Survived'].mean() * 100:.2f}% ({below_3['Survived'].sum()}/{len(below_3)})")
print(f"Family size == 3 : {equal_3['Survived'].mean() * 100:.2f}% ({equal_3['Survived'].sum()}/{len(equal_3)})")

## 7. Key insights (generated from the data)

In [ ]:
print(f"1. Female survivors ({female_survived}) outnumbered male survivors ({male_survived}); average survivor age was {avg_survivor_age:.2f} years.")
print(f"2. Highest survival rate by embarkation port was Cherbourg (C) at {embarked_rate['C']:.2f}%, compared to Queenstown (Q) at {embarked_rate['Q']:.2f}% and Southampton (S) at {embarked_rate['S']:.2f}%.")
print(f"3. Middle-aged women (30-45) had a {mid_women['Survived'].mean() * 100:.2f}% survival rate vs {mid_men['Survived'].mean() * 100:.2f}% for middle-aged men (30-45).")
print(f"4. Passengers traveling with family survived at {with_family['Survived'].mean() * 100:.2f}% vs {solo['Survived'].mean() * 100:.2f}% for solo travelers.")
print(f"5. Families of size 3 achieved the highest survival rate ({equal_3['Survived'].mean() * 100:.2f}%), whereas families larger than 3 dropped to {above_3['Survived'].mean() * 100:.2f}%.")

## 8. Summary & takeaways
- **Women and children first:** Middle-aged women (`30–45`) and children (`< 15`) experienced markedly higher survival rates than adult men and elderly passengers (`> 60`).
- **Port of embarkation matters:** Passengers boarding at Cherbourg (`C`) had the strongest survival rate, reflecting a higher share of first-class passengers.
- **Small family advantage:** Traveling in a small family group (especially `family_size == 3`) improved survival odds compared to traveling alone or in large families (`> 3`).

> **Remember:** correlation does not prove causation. This exploratory analysis highlights historical patterns in the dataset.

## 9. Save the cleaned data

In [ ]:
# 10. Save the clean file
df.to_csv("cleaned_titanic.csv", index=False)
print("Cleaning done! Saved to cleaned_titanic.csv with", len(df), "rows")